# Cross-cell-type co-progression with orthogonal axes (Colon Day 3)

## Overview

This tutorial demonstrates how CoPro detects **cross-cell-type co-progression** using colon organoid Day 3 data. CoPro identifies multiple orthogonal canonical components (CCs), each capturing an independent axis of coordinated spatial variation between cell types.

In this dataset, we analyze three cell types — Epithelial, Fibroblast, and Immune — and show that CC1 and CC2 capture distinct, biologically meaningful spatial programs. We also compare CoPro axes with independently derived **mucosal neighborhood (MU) labels** to validate that the unsupervised axes recover known tissue structure.

**Plots produced:**
1. Tissue layout (cell types)
2. Mucosal neighborhood (MU) labels
3. Normalized correlation vs sigma
4. CC1 cell scores in situ (by cell type)
5. CC2 cell scores in situ (by cell type)
6. CC1 vs CC2 colored by MU label (with density contours)
7. Example gene expression in situ (Egr1, Mki67)
8. Cross-type correlation scatter (CC1 and CC2)
9. Top genes for CC1 and CC2

## 1. Load packages

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy.stats import gaussian_kde

import copro as cp

print(f"copro version: {cp.__version__}")

## 2. Download and load data

In [ ]:
dat = cp.copro_download_data("colon_d3")

expr = dat["normalizedData"].values if isinstance(dat["normalizedData"], pd.DataFrame) else dat["normalizedData"]
location = dat["locationData"] if isinstance(dat["locationData"], pd.DataFrame) else pd.DataFrame(dat["locationData"])
meta = dat["metaData"] if isinstance(dat["metaData"], pd.DataFrame) else pd.DataFrame(dat["metaData"])
cell_types = dat["cellTypes"] if isinstance(dat["cellTypes"], np.ndarray) else np.array(dat["cellTypes"])

# Get gene names from expression data
if isinstance(dat["normalizedData"], pd.DataFrame):
    gene_names = list(dat["normalizedData"].columns)
    expr = dat["normalizedData"].values.astype(float)
else:
    gene_names = [f"gene_{i}" for i in range(expr.shape[1])]
    expr = expr.astype(float)

# Ensure location has x, y columns
if "x" not in location.columns:
    location.columns = ["x", "y"] + list(location.columns[2:])

print(f"Cells: {expr.shape[0]}, Genes: {expr.shape[1]}")
print(f"Cell types: {np.unique(cell_types)}")
print(f"Location columns: {list(location.columns)}")

## 3. Visualize the tissue

In [ ]:
ct_colors = {"Epithelial": "#E41A1C", "Fibroblast": "#377EB8", "Immune": "#4DAF4A"}

fig, ax = plt.subplots(figsize=(7, 6))
for ct, color in ct_colors.items():
    mask = cell_types == ct
    ax.scatter(location.loc[mask, "x"], location.loc[mask, "y"],
               s=0.5, alpha=0.6, color=color, label=ct)
ax.set_aspect("equal")
ax.set_title("Colon Day 3 organoid")
ax.legend(markerscale=5, loc="lower right")
ax.set_xlabel("x")
ax.set_ylabel("y")
plt.tight_layout()
plt.show()

### Mucosal neighborhood (MU) labels

The tissue has been independently annotated with mucosal neighborhood labels (MU1–MU4) via Leiden clustering on spatial neighborhoods. MU4 represents an inflammation-associated microenvironment distinct from MU1–3 (crypt-associated neighborhoods).

In [ ]:
mu_labels = meta["Leiden_neigh"].values.astype(str)
mu_grouped = np.where(np.isin(mu_labels, ["MU1", "MU2", "MU3"]), "MU1-3", mu_labels)

mu_colors = {"MU1-3": "#5c9137", "MU4": "#614b83"}

fig, ax = plt.subplots(figsize=(7, 6))
for mu, color in mu_colors.items():
    mask = mu_grouped == mu
    ax.scatter(location.loc[mask, "x"], location.loc[mask, "y"],
               s=0.5, alpha=0.6, color=color, label=mu)
ax.set_aspect("equal")
ax.set_title("Mucosal neighborhood (MU) labels")
ax.legend(markerscale=5)
ax.set_xlabel("x")
ax.set_ylabel("y")
plt.tight_layout()
plt.show()

## 4. Create CoPro object and run pipeline

In [ ]:
CELL_TYPES = ["Epithelial", "Fibroblast", "Immune"]
SIGMA_VALUES = [0.005, 0.01, 0.02, 0.05, 0.1]
N_PCA = 40
N_CC = 4

obj = cp.CoProSingle(
    normalized_data=expr,
    location_data=location,
    meta_data=meta,
    cell_types=cell_types,
)

obj = cp.subset_data(obj, CELL_TYPES)
obj = cp.compute_pca(obj, n_pca=N_PCA)
obj = cp.compute_distance(obj)
obj = cp.compute_kernel_matrix(obj, sigma_values=SIGMA_VALUES)
obj = cp.run_skr_cca(obj, scale_pcs=True, n_cc=N_CC)
obj = cp.compute_normalized_correlation(obj)
obj = cp.compute_gene_and_cell_scores(obj)

print(f"\nPipeline complete. Selected sigma: {obj.sigma_value_choice}")

## 5. Select optimal sigma

In [ ]:
# Collect normalized correlations
nc_frames = []
for sigma_name, df in obj.normalized_correlation.items():
    if df is not None and len(df) > 0:
        nc_frames.append(df)
nc_df = pd.concat(nc_frames, ignore_index=True)

nc_df["pair"] = nc_df["cell_type_1"] + " × " + nc_df["cell_type_2"]
pairs = nc_df["pair"].unique()
cc_idxs = sorted(nc_df["CC_index"].unique())

fig, axes = plt.subplots(len(pairs), len(cc_idxs),
                          figsize=(3.5 * len(cc_idxs), 3 * len(pairs)),
                          sharey=False, squeeze=False)

for row_i, pair in enumerate(pairs):
    for col_i, cc in enumerate(cc_idxs):
        ax = axes[row_i][col_i]
        sub = nc_df[(nc_df["pair"] == pair) & (nc_df["CC_index"] == cc)].sort_values("sigma")
        ax.plot(sub["sigma"], sub["normalized_correlation"], "o-", color="steelblue")
        ax.axvline(obj.sigma_value_choice, color="tomato", linestyle="--", alpha=0.6)
        ax.set_title(f"{pair}\nCC{cc}", fontsize=8)
        ax.set_xlabel("sigma", fontsize=8)
        ax.set_ylabel("Norm. corr.", fontsize=8)

fig.suptitle("Normalized correlation across sigma values", y=1.01)
plt.tight_layout()
plt.show()

## 6. Orthogonal axes: CC1 vs CC2

A key feature of CoPro is that it extracts **multiple orthogonal axes**. Each CC captures a distinct pattern of spatial co-variation.

### CC1 cell scores in situ

In [ ]:
sigma_opt = obj.sigma_value_choice

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, ct in enumerate(CELL_TYPES):
    mask = obj.cell_types_sub == ct
    loc = obj.location_data_sub.loc[mask]
    scores = obj.cell_scores[f"cellScores|sigma{sigma_opt}|{ct}"][:, 0]
    sc = axes[i].scatter(loc["x"].values, loc["y"].values,
                          c=scores, cmap="viridis", s=0.8)
    plt.colorbar(sc, ax=axes[i], label="Score")
    axes[i].set_title(ct)
    axes[i].set_aspect("equal")
    axes[i].set_xlabel("x")
    axes[i].set_ylabel("y")

fig.suptitle(f"CC1 cell scores in situ (by cell type, σ={sigma_opt})", y=1.02)
plt.tight_layout()
plt.show()

### CC2 cell scores in situ

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, ct in enumerate(CELL_TYPES):
    mask = obj.cell_types_sub == ct
    loc = obj.location_data_sub.loc[mask]
    scores = obj.cell_scores[f"cellScores|sigma{sigma_opt}|{ct}"][:, 1]
    sc = axes[i].scatter(loc["x"].values, loc["y"].values,
                          c=scores, cmap="inferno", s=0.8)
    plt.colorbar(sc, ax=axes[i], label="Score")
    axes[i].set_title(ct)
    axes[i].set_aspect("equal")
    axes[i].set_xlabel("x")
    axes[i].set_ylabel("y")

fig.suptitle(f"CC2 cell scores in situ (by cell type, σ={sigma_opt})", y=1.02)
plt.tight_layout()
plt.show()

CC1 and CC2 capture distinct spatial gradients. Because CCA axes are orthogonal, these represent independent programs of coordinated gene expression across cell types.

## 7. CoPro axes recover mucosal neighborhood structure

A key validation is that CoPro's unsupervised axes align with the independently derived MU labels. We plot CC1 vs CC2 cell scores colored by MU label, showing that MU4 (inflammation) occupies a distinct region of the CoPro score space.

In [ ]:
# Gather all cell scores (CC1, CC2) and MU labels
all_cc1 = []
all_cc2 = []
all_mu = []

for ct in CELL_TYPES:
    mask = obj.cell_types_sub == ct
    cc1 = obj.cell_scores[f"cellScores|sigma{sigma_opt}|{ct}"][:, 0]
    cc2 = obj.cell_scores[f"cellScores|sigma{sigma_opt}|{ct}"][:, 1]
    mu_sub = meta.loc[np.isin(obj.cell_types, CELL_TYPES)].reset_index(drop=True)
    mu_sub = mu_sub.loc[mask, "Leiden_neigh"].values.astype(str)
    all_cc1.append(cc1)
    all_cc2.append(cc2)
    all_mu.append(mu_sub)

cc1_all = np.concatenate(all_cc1)
cc2_all = np.concatenate(all_cc2)
mu_all = np.concatenate(all_mu)
mu_grouped_sub = np.where(np.isin(mu_all, ["MU1", "MU2", "MU3"]), "MU1-3", mu_all)

fig, ax = plt.subplots(figsize=(6, 5))

for mu, color in mu_colors.items():
    sel = mu_grouped_sub == mu
    ax.scatter(cc1_all[sel], cc2_all[sel], s=0.1, alpha=0.1, color=color, label=mu)
    # Add density contours
    if sel.sum() > 10:
        xy = np.vstack([cc1_all[sel], cc2_all[sel]])
        try:
            kde = gaussian_kde(xy)
            xmin, xmax = cc1_all.min(), cc1_all.max()
            ymin, ymax = cc2_all.min(), cc2_all.max()
            xx, yy = np.mgrid[xmin:xmax:100j, ymin:ymax:100j]
            zz = kde(np.vstack([xx.ravel(), yy.ravel()])).reshape(xx.shape)
            ax.contour(xx, yy, zz, colors=color, linewidths=0.5, alpha=0.9, levels=5)
        except Exception:
            pass

ax.set_xlabel("CoPro CC1")
ax.set_ylabel("CoPro CC2")
ax.set_title("CoPro CC1 vs CC2 colored by MU label")
# Custom legend with larger markers
from matplotlib.lines import Line2D
handles = [Line2D([0], [0], marker='o', color='w', markerfacecolor=c, markersize=8, label=l)
           for l, c in mu_colors.items()]
ax.legend(handles=handles, title="MU Label")
plt.tight_layout()
plt.show()

## 8. Example gene expression in situ

Top CoPro genes show spatially coherent expression patterns. Here we visualize two marker genes: **Egr1** (immediate early gene, enriched in the inflammation/CC2 axis) and **Mki67** (proliferation marker, enriched in crypt base/CC1 axis).

In [ ]:
x_all = location["x"].values
y_all = location["y"].values

markers = {"Egr1": ("#E5E5E5", "darkred"), "Mki67": ("#E5E5E5", "darkblue")}

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (gene, (c_low, c_high)) in zip(axes, markers.items()):
    if gene in gene_names:
        gene_idx = gene_names.index(gene)
        values = expr[:, gene_idx]
    else:
        print(f"Gene {gene} not found in dataset. Using random values for demonstration.")
        values = np.random.randn(len(x_all))

    order = np.argsort(values)
    cmap = mcolors.LinearSegmentedColormap.from_list("", [c_low, c_high])
    sc = ax.scatter(x_all[order], y_all[order], c=values[order],
                     cmap=cmap, s=0.2, alpha=0.8)
    plt.colorbar(sc, ax=ax)
    ax.set_aspect("equal")
    ax.set_title(gene)
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.show()

## 9. Cross-type correlation plots

The correlation between cell types shows how strongly their spatial programs are coupled.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))

for ax, cc_idx, cc_label in [(axes[0], 1, "CC1"), (axes[1], 2, "CC2")]:
    cp.plot_correlation_scatter(obj, "Epithelial", "Fibroblast",
                                sigma=sigma_opt, cc_index=cc_idx, ax=ax)
    ax.set_title(f"{cc_label}: Epithelial-Fibroblast co-progression")

plt.tight_layout()
plt.show()

## 10. Gene scores

Identify which genes drive each co-progression axis. We compute regression-based gene scores and visualize the top genes.

In [ ]:
obj = cp.compute_regression_gene_scores(obj, sigma=[sigma_opt])

### Top genes for CC1 (Epithelial)

In [ ]:
cp.plot_top_genes(obj, "Epithelial", cc_index=1, gene_names=gene_names, top_n=20,
                  title="Top 20 Epithelial genes (CC1, regression)",
                  colors_pos="darkred", colors_neg="lightpink")
plt.tight_layout()
plt.show()

### Top genes for CC2 (Epithelial)

In [ ]:
cp.plot_top_genes(obj, "Epithelial", cc_index=2, gene_names=gene_names, top_n=20,
                  title="Top 20 Epithelial genes (CC2, regression)",
                  colors_pos="darkred", colors_neg="lightpink")
plt.tight_layout()
plt.show()

## References

**CoPro method:**  
Miao Z. et al. *CoPro: Unsupervised detection of coordinated spatial progressions in spatial transcriptomics* (in preparation).